# Web Scraping Notebook (AutoScout24 Example)

This notebook is written for students who are scraping a website for the first time.
Every step is small and explicit, and each code cell has comments that explain exactly what is happening.

What we will extract from each listing:
- make
- model
- mileage
- price
- registration date
- fuel type

Important notes for beginners:
- Always check the website rules and robots.txt before scraping.
- Start with a small number of pages while you learn.
- If a page is blocked or requires JavaScript, you may not see the listings.



## Step 0: Install the libraries (only once)

We use three Python libraries:
- `requests` to download web pages
- `beautifulsoup4` to parse HTML
- `pandas` to save data into a table

If you already installed them, you can skip running this cell next time.



In [ ]:
%pip install requests beautifulsoup4 pandas


## Step 1: Import the libraries

We now import the libraries we just installed. This makes their functions available in the notebook.
Each import is commented so you remember what it is used for.



In [ ]:
import requests  # Download web pages with HTTP requests
from bs4 import BeautifulSoup  # Parse HTML into searchable elements
import pandas as pd  # Store and export data as a table
import time  # Pause between requests to be polite


## Step 2: Define the search parameters

We choose a small list of brands and a short page range. This is enough to learn the process
and keeps the load on the website low. We also create a base URL and a simple User-Agent header
so the request looks like it comes from a normal browser.



In [ ]:
brands = ['bmw', 'mercedes-benz', 'audi']  # Example brands to query
start_page = 1  # First results page to request
end_page = 3  # Last page (inclusive) for this demo
base_url = 'https://www.autoscout24.com/lst/{brand}'  # URL template per brand
headers = {
    'User-Agent': 'Mozilla/5.0 (educational example)'  # Simple browser-like header
}  # Some sites reject requests without a User-Agent


## Step 3: Build one example URL

Before looping over many pages, we build one example URL to verify the format.
This makes debugging easier because we can test a single page first.



In [ ]:
example_url = base_url.format(brand='bmw') + '?atype=C&desc=0&page=1'  # Single test URL
example_url  # Display the URL to verify it looks correct


## Step 4: Download the example page

We download the page using a `Session`, which keeps cookies and headers between requests.
We print the HTTP status code and stop if the page did not load correctly.



In [ ]:
session = requests.Session()  # Reuse cookies/headers across requests
response = session.get(example_url, headers=headers, timeout=10)  # Download the page
status_code = response.status_code  # HTTP status code returned by the server
print('Status code:', status_code)  # Show the result
if status_code != 200:  # Stop if the response is not OK
    raise ValueError(f'Request failed with status code {status_code}')  # Explain the error


## Step 5: Parse the HTML

The response is raw HTML text. We convert it into a BeautifulSoup object so we can search it
for specific elements like listing cards.



In [ ]:
html_text = response.text  # Raw HTML as text
soup = BeautifulSoup(html_text, 'html.parser')  # Create the parser


## Step 6: Check the page title

We read the `<title>` tag. If the title looks correct, the page probably loaded.
If it is missing or generic, the website might be blocking us or loading data with JavaScript.



In [ ]:
title_tag = soup.find('title')  # Look for the title tag
page_title = title_tag.get_text(strip=True) if title_tag else 'No title found'  # Safe extraction
print('Page title:', page_title)  # Display the title


## Step 7: Inspect one listing card (checkpoint)

Before scraping many pages, we inspect one listing card to learn the HTML structure.
We print a short HTML snippet so you can see where the data is stored.



In [ ]:
# Checkpoint: inspect one listing card (HTML snippet)
sample_card = soup.select_one('article[data-testid="decluttered-list-item"]')
if sample_card:
    print(sample_card.prettify()[:1500])  # Show the first 1500 chars
else:
    print('No listing card found on the page.')


## Step 8: Extract data from all pages

We define a small helper function that reads values from the HTML `data-*` attributes on each card.
Then we loop through the brands and pages, collect all listings, and build a table.
We also add short pauses between requests to be polite.



In [ ]:
def parse_listing_card(card):
    # Read data directly from HTML data-* attributes on the card
    return {
        'make': card.get('data-make'),  # Brand name
        'model': card.get('data-model'),  # Model name
        'mileage': card.get('data-mileage'),  # Mileage in km (as string)
        'price': card.get('data-price'),  # Price in EUR (as string)
        'registration': card.get('data-first-registration'),  # First registration date
        'fuel': card.get('data-fuel-type'),  # Fuel type code
    }

results = []  # List where we will store results
for brand in brands:  # Iterate over each brand
    for page in range(start_page, end_page + 1):  # Iterate over each page
        url = base_url.format(brand=brand) + f'?atype=C&desc=0&page={page}'  # Build the URL
        response = session.get(url, headers=headers, timeout=10)  # Download the page
        if response.status_code != 200:  # Skip if the request fails
            print(f'Error {response.status_code} on {url}')  # Simple warning
            continue  # Move to the next page
        soup = BeautifulSoup(response.text, 'html.parser')  # Parse the HTML

        cards = soup.select('article[data-testid="decluttered-list-item"]')  # Listing cards
        if not cards:
            print(f'No listings found on {url} (page may be blocked or rendered with JS).')
            continue

        for card in cards:
            row = parse_listing_card(card)  # Extract fields
            row['brand'] = brand  # Add brand context
            row['page'] = page  # Add page context
            results.append(row)  # Append to results

        time.sleep(1)  # Pause to avoid overloading the server

results_df = pd.DataFrame(results)  # Convert to a DataFrame
results_df.head()  # Show the first rows


## Step 9: Save the results to the repo data folder

We save the DataFrame into the `data/raw` folder at the root of the repository.
We include a timestamp in the filename so each run creates a new file.



In [ ]:
from datetime import datetime
from pathlib import Path

# Resolve repo root (walk up until we find the data folder)
repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / 'data').exists():
    repo_root = repo_root.parent

output_dir = repo_root / 'data' / 'raw'  # Output directory at repo root
output_dir.mkdir(parents=True, exist_ok=True)  # Create if missing
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')  # Versioned filename
output_path = output_dir / f'autoscout24_listings_{timestamp}.csv'  # Output file path
results_df.to_csv(output_path, index=False)  # Save without the index column
print('Saved to', output_path)  # Confirmation message
